# Clasificación de género musical peruano con AST (visión sobre espectrogramas)

**Géneros:** chicha, cumbia amazónica, música criolla, huayno, rock
**Dataset:** [`pollitoconpapass/dataset-peru-shazam`](https://huggingface.co/datasets/pollitoconpapass/dataset-peru-shazam) (500 canciones, 100 por género)
**Objetivo:** dado un audio de **10 a 15 segundos**, predecir el género.

## Paradigma (v2)

En lugar de features manuales + CNN desde cero, tratamos el audio como una **imagen** (su log-espectrograma Mel)
y clasificamos esa imagen con **transfer learning**:

- **Backbone:** [AST](https://arxiv.org/abs/2104.01778) (*Audio Spectrogram Transformer*), pre-entrenado en **AudioSet**
  (millones de clips de audio). AST es literalmente un ViT aplicado al espectrograma: un modelo de visión sobre la imagen del audio.
- Cada canción se corta en **clips de ~10 s sin solape** repartidos por toda la canción → ~6 000–8 000 ejemplos
  de entrenamiento (5–10× más que un solo clip por canción).
- **Split por canción**: todos los clips de una misma canción van al mismo conjunto (sin fuga).
- **SpecAugment** durante el entrenamiento (máscaras aleatorias de tiempo y frecuencia).
- `n_mels = 128` para separar mejor las bandas donde se confunden **chicha** y **cumbia amazónica**.


## 0. Instalación e imports

In [ ]:
!pip install -q transformers torchaudio librosa datasets soundfile scikit-learn matplotlib tqdm joblib


In [ ]:
import os, io, warnings, joblib
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset, Audio

import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")

SEED = 42
SR = 16000                      # AST trabaja a 16 kHz
CLIP_SEC = 10.24                # 1024 frames x 10 ms = la ventana que espera AST
CLIP_SAMPLES = int(SR * CLIP_SEC)
CLIP_FRAMES = 1024              # frames (tiempo) del espectrograma por clip
N_MELS = 128                    # resolución en frecuencia (chicha vs cumbia amazónica)
CACHE_FEATS = "ast_feats.npy"   # espectrogramas por clip (float16)
CACHE_META  = "clip_meta.npz"   # etiquetas y relación clip -> canción

MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"   # AST pre-entrenado en AudioSet

np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| librosa", librosa.__version__, "| torch", torch.__version__)


## 1. Carga del dataset y emparejamiento correcto de etiquetas

El `catalog.json` y el dataset de audio **no tienen por qué estar en el mismo orden**
Emparejar por `filename`.

In [ ]:
catalog_dataset = load_dataset("pollitoconpapass/dataset-peru-shazam", data_files="catalog.json")
json_content = catalog_dataset['train'][0]
df_metadata = pd.DataFrame.from_dict(json_content, orient='index')

df_validos = df_metadata[(df_metadata['status'] == 'downloaded') & (df_metadata['error'].isnull())].reset_index(drop=True)
df_validos['genre'] = df_validos['genre'].astype(str).str.strip().str.lower()

print(len(df_validos), "canciones válidas en el catálogo")
print(df_validos['genre'].value_counts())

In [ ]:
# Audios SIN decodificar (solo rutas/bytes): así no cargamos ~10 GB de audio en RAM.
ds_audio = load_dataset("pollitoconpapass/dataset-peru-shazam")['train']
ds_audio = ds_audio.cast_column("audio", Audio(decode=False))
audio_refs = ds_audio['audio']            # lista de dicts {'path':..., 'bytes':...}

archivos = [os.path.basename(a['path']) for a in audio_refs]
genero_por_archivo   = dict(zip(df_validos['filename'], df_validos['genre']))
duracion_por_archivo = dict(zip(df_validos['filename'], df_validos['duration_seconds']))

faltan = [a for a in archivos if a not in genero_por_archivo]
print(f"Audios en el dataset: {len(archivos)} | sin etiqueta en el catálogo: {len(faltan)}")
assert len(faltan) == 0, f"Archivos sin etiqueta, ejemplo: {faltan[:5]}"


In [ ]:
genero_correcto = np.array([genero_por_archivo[a] for a in archivos])     # emparejado por nombre de archivo
genero_antiguo  = df_validos['genre'].values[:len(archivos)]              # emparejado por posición (versión anterior)

coincide = (genero_correcto == genero_antiguo).mean()
print(f"\nCoincidencia etiquetas por POSICIÓN vs por ARCHIVO: {coincide:.1%}")

In [ ]:
# Verificación de la duración real del audio debe parecerse a la duración del catálogo
dif = []
for a in audio_refs[:25]:
    fn = os.path.basename(a['path'])
    dur_real = librosa.get_duration(path=a['path'])
    dif.append(abs(dur_real - duracion_por_archivo[fn]))
print(f"Diferencia media de duración (real vs catálogo) en 25 canciones: {np.mean(dif):.1f} s  (esperado: ~0–2 s)")

## 2. Clips de 10 s y su espectrograma (input de AST)

Para cada canción tomamos **clips de 10.24 s sin solape** entre el 5 % y el 95 % de la duración
(se evitan intros/finales). Una canción de 3–4 min aporta ~15–20 clips: pasamos de 500 canciones
a **~6 000–8 000 ejemplos de entrenamiento**.

De cada clip calculamos su **log-espectrograma Mel (128 bins × 1024 frames)** con el pre-procesado
oficial de AST (`ASTFeatureExtractor`), así el backbone recibe exactamente lo que vio en AudioSet.
Todo se cachea en disco para no re-decodificar los ~10 GB de audio en cada ejecución.


In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

def cargar_audio_16k(ref):
    """Decodifica un audio del dataset a mono 16 kHz (el SR que espera AST)."""
    src = ref['path'] if ref.get('path') and os.path.exists(ref['path']) else io.BytesIO(ref['bytes'])
    y, _ = librosa.load(src, sr=SR, mono=True)
    return y

def inicios_clips(n_samples, clip=CLIP_SAMPLES):
    """Inicios (en muestras) de clips de 10.24 s SIN solape entre el 5 % y el 95 % de la canción."""
    ini, fin = int(0.05 * n_samples), int(0.95 * n_samples) - clip
    if fin < ini:                        # canción más corta que un clip: usamos el inicio
        return np.array([0])
    return np.arange(ini, fin + 1, clip)

def espectrograma_ast(waveform):
    """Log-mel (1024, 128) normalizado, idéntico al pre-procesado oficial de AST."""
    return feature_extractor(waveform, sampling_rate=SR)['input_values'][0].astype(np.float16)

def ventanas_inferencia(y, clip=CLIP_SAMPLES, min_sobra=0.5):
    """Ventanas de 10.24 s desde el inicio del clip de entrada (10-15 s).
    Si al final sobra >= 0.5 s, se añade una última ventana rellena con silencio."""
    if len(y) < clip:
        return [np.pad(y, (0, clip - len(y)))]
    inis = list(range(0, len(y) - clip + 1, clip))
    if len(y) - (inis[-1] + clip) >= int(min_sobra * SR):
        inis.append(len(y) - clip)
    return [y[s:s + clip] for s in inis]


In [ ]:
le = LabelEncoder().fit(genero_correcto)
y_song = le.transform(genero_correcto)          # una etiqueta por canción
print("Clases:", [str(c) for c in le.classes_])

if os.path.exists(CACHE_FEATS) and os.path.exists(CACHE_META):
    print("Cargando espectrogramas desde caché…")
    X = np.load(CACHE_FEATS)                                  # (n_clips, 1024, 128) float16
    meta = np.load(CACHE_META, allow_pickle=True)
    y_clip, clip_song = meta['y_clip'], meta['clip_song']
    assert list(meta['classes']) == list(le.classes_) and list(meta['archivos']) == archivos, \
        "La caché no corresponde a este dataset: bórrala."
else:
    Xs, Yc, Cs = [], [], []
    for i, ref in enumerate(tqdm(audio_refs, desc="Extrayendo espectrogramas")):
        y = cargar_audio_16k(ref)
        if len(y) < CLIP_SAMPLES:
            y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
        for s in inicios_clips(len(y)):
            Xs.append(espectrograma_ast(y[s:s + CLIP_SAMPLES]))
            Yc.append(y_song[i]); Cs.append(i)
    X = np.stack(Xs)
    y_clip, clip_song = np.array(Yc), np.array(Cs)
    np.save(CACHE_FEATS, X)
    np.savez_compressed(CACHE_META, y_clip=y_clip, clip_song=clip_song,
                        classes=le.classes_, archivos=np.array(archivos))

print(f"Clips: {len(y_clip)} | forma de X: {X.shape} (frames × mels) | canciones: {len(y_song)}")


## 3. Split por canción (train / val / test)

Todos los clips de una misma canción van al mismo conjunto. Si no, el modelo "reconoce la canción"
en lugar de aprender el género y la métrica sale inflada.


In [ ]:
ids = np.arange(len(y_song))
ids_trval, ids_test = train_test_split(ids, test_size=0.20, stratify=y_song, random_state=SEED)
ids_train, ids_val  = train_test_split(ids_trval, test_size=0.15, stratify=y_song[ids_trval], random_state=SEED)

def mask(ids_): return np.isin(clip_song, ids_)
m_train, m_val, m_test = mask(ids_train), mask(ids_val), mask(ids_test)

print(f"Canciones -> train {len(ids_train)} | val {len(ids_val)} | test {len(ids_test)}")
print(f"Clips     -> train {m_train.sum()} | val {m_val.sum()} | test {m_test.sum()}")


## 4. Evaluación

Sobre las **canciones de test**:

- **Clip (10 s):** accuracy de cada clip individual.
- **Clip ≈15 s:** promedio de las probabilidades de 2 clips al azar de la misma canción (simula el caso de uso; 30 repeticiones).
- **Canción:** promedio de las probabilidades de todos sus clips.


In [ ]:
def evaluar(nombre, P_fn, n_clip_clips=2, n_rep=30):
    P = P_fn(m_test)
    yt, st = y_clip[m_test], clip_song[m_test]
    acc_clip = accuracy_score(yt, P.argmax(1))

    rg = np.random.default_rng(SEED)
    acc_15s, acc_song = [], []
    for s in np.unique(st):
        Ps = P[st == s]; label = yt[st == s][0]
        acc_song.append(Ps.mean(0).argmax() == label)
        k = min(n_clip_clips, len(Ps))
        acc_15s.append(np.mean([Ps[rg.choice(len(Ps), k, replace=False)].mean(0).argmax() == label
                                for _ in range(n_rep)]))
    res = dict(modelo=nombre, clip_10s=acc_clip, clip_15s=np.mean(acc_15s), cancion=np.mean(acc_song))
    print(f"{nombre:28s} | clip 10s {acc_clip:.3f} | clip≈15s {res['clip_15s']:.3f} | canción {res['cancion']:.3f}")
    return P, res


## 5. Transfer learning: AST (modelo de visión sobre el espectrograma)

Cargamos el **AST pre-entrenado en AudioSet** y sustituimos su cabeza de clasificación
(527 clases → nuestras 5). El backbone se **fine-tunea completo** con LR bajo (1e-5) y la cabeza
nueva con LR alto (1e-3): con ~6 000+ clips hay datos de sobra para afinar todos los pesos.

Durante el entrenamiento aplicamos **SpecAugment**: 2 máscaras de tiempo + 2 de frecuencia por
clip, rellenadas con 0 (la media global tras la normalización de AST).


In [ ]:
n_classes = len(le.classes_)
model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID, num_labels=n_classes, ignore_mismatched_sizes=True)   # cabeza nueva, inicializada al azar
model.config.id2label = {i: c for i, c in enumerate(le.classes_)}
model.config.label2id = {c: i for i, c in enumerate(le.classes_)}
model.to(device)

def spec_augment(x):                    # x: (1024, 128) frames × mels, ya normalizado (media 0)
    for _ in range(2):                  # máscaras de tiempo (franjas verticales)
        t = int(torch.randint(10, 60, (1,))); t0 = int(torch.randint(0, CLIP_FRAMES - t, (1,)))
        x[t0:t0 + t, :] = 0.0
    for _ in range(2):                  # máscaras de frecuencia (bandas horizontales)
        f = int(torch.randint(5, 20, (1,))); f0 = int(torch.randint(0, N_MELS - f, (1,)))
        x[:, f0:f0 + f] = 0.0
    return x

class ClipDataset(torch.utils.data.Dataset):
    def __init__(self, m, train):
        self.idx = np.where(m)[0]; self.train = train
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        x = X[self.idx[i]].astype(np.float32)
        if self.train:
            x = spec_augment(x)
        return torch.from_numpy(x), int(y_clip[self.idx[i]])

BATCH = 16
loader = lambda m, train: torch.utils.data.DataLoader(
    ClipDataset(m, train), batch_size=BATCH, shuffle=train, num_workers=2,
    pin_memory=(device == "cuda"))
train_dl, val_dl = loader(m_train, True), loader(m_val, False)

head     = [p for n, p in model.named_parameters() if 'classifier' in n]
backbone = [p for n, p in model.named_parameters() if 'classifier' not in n]
opt = torch.optim.AdamW([
    dict(params=backbone, lr=1e-5, weight_decay=0.01),
    dict(params=head,     lr=1e-3, weight_decay=0.01),
])


In [ ]:
EPOCHS, PATIENCE = 20, 4
best_val, best_state, history, bad = np.inf, None, [], 0
scaler = torch.amp.GradScaler(enabled=(device == "cuda"))

for epoch in range(1, EPOCHS + 1):
    model.train(); tl = tc = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            logits = model(input_values=xb).logits
        loss = torch.nn.functional.cross_entropy(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        tl += loss.item() * len(yb); tc += (logits.argmax(1) == yb).sum().item()

    model.eval(); vl = vc = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(input_values=xb).logits
            vl += torch.nn.functional.cross_entropy(logits, yb).item() * len(yb)
            vc += (logits.argmax(1) == yb).sum().item()

    n_tr, n_va = m_train.sum(), m_val.sum()
    history.append((tl / n_tr, tc / n_tr, vl / n_va, vc / n_va))
    print(f"Época {epoch:2d} | train loss {tl/n_tr:.3f} acc {tc/n_tr:.3f} | "
          f"val loss {vl/n_va:.3f} acc {vc/n_va:.3f}")

    if vl < best_val:
        best_val, bad = vl, 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(best_state)

h = np.array(history)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(h[:, 0], label='train'); ax[0].plot(h[:, 2], label='val'); ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(h[:, 1], label='train'); ax[1].plot(h[:, 3], label='val'); ax[1].set_title('Accuracy'); ax[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
@torch.no_grad()
def predecir_clips(m, batch=64):
    model.eval()
    dl = torch.utils.data.DataLoader(ClipDataset(m, False), batch_size=batch)
    return np.concatenate([
        torch.softmax(model(input_values=xb.to(device)).logits, dim=1).cpu().numpy()
        for xb, _ in dl])

P_test, res_ast = evaluar("AST fine-tuned", predecir_clips)


## 6. Matriz de confusión: ¿dónde se confunde la chicha?

Nos interesa especialmente la confusión **chicha ↔ cumbia amazónica**: comparten guitarra eléctrica,
estructura de cumbia y origen (la chicha es cumbia andina). Con 128 bins de Mel el modelo tiene
resolución de sobra para separarlas; aquí comprobamos cuánto se cruzan aún.


In [ ]:
yt = y_clip[m_test]
cm = confusion_matrix(yt, P_test.argmax(1))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(cmap='Blues', xticks_rotation=45)
plt.title('Confusión (clips de 10 s) – AST'); plt.tight_layout(); plt.show()

print(classification_report(yt, P_test.argmax(1), target_names=le.classes_))

i_ch = [i for i, c in enumerate(le.classes_) if 'chicha' in c][0]
i_ca = [i for i, c in enumerate(le.classes_) if 'cumbia' in c][0]
cm_par = cm[np.ix_([i_ch, i_ca], [i_ch, i_ca])]
print(f"Submatriz {le.classes_[i_ch]} vs {le.classes_[i_ca]}  [filas: real, columnas: predicho]")
print(cm_par)
print(f"Confusión mutua: {cm_par[0, 1] + cm_par[1, 0]} de {cm_par.sum()} clips de estos dos géneros")


## 7. Inferencia sobre un clip de 10–15 s

El clip (hasta 15 s) se parte en ventanas de 10.24 s desde el inicio; si sobra audio al final, se añade una
última ventana rellena con silencio. Se predice cada ventana y se **promedian las probabilidades**.


In [ ]:
def predecir_genero(ruta_audio, max_seg=15):
    y, _ = librosa.load(ruta_audio, sr=SR, mono=True, duration=max_seg)
    if len(y) < CLIP_SAMPLES:
        y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
    clips = ventanas_inferencia(y)
    X_inf = np.stack([feature_extractor(c, sampling_rate=SR)['input_values'][0]
                      for c in clips]).astype(np.float32)
    model.eval()
    with torch.no_grad():
        logits = model(input_values=torch.from_numpy(X_inf).to(device)).logits
    proba = torch.softmax(logits, dim=1).mean(0).cpu().numpy()
    orden = np.argsort(proba)[::-1]
    return str(le.classes_[orden[0]]), {str(le.classes_[i]): round(float(proba[i]), 3) for i in orden}

# Ejemplo: recortamos 12 s del medio de una canción de TEST y la clasificamos
import soundfile as sf
i_demo = int(ids_test[0])
y_full = cargar_audio_16k(audio_refs[i_demo])
ini = len(y_full) // 2
sf.write("clip_prueba.wav", y_full[ini: ini + 12 * SR], SR)

pred, probas = predecir_genero("clip_prueba.wav")
print("Género real:", le.classes_[y_song[i_demo]])
print("Predicción :", pred)
print(probas)


## 8. Guardado del modelo en Hugging Face

In [ ]:
from huggingface_hub import HfApi

os.makedirs("modelo_peru_genre", exist_ok=True)
model.save_pretrained("modelo_peru_genre")          # pesos + config (con id2label ya incluido)
feature_extractor.save_pretrained("modelo_peru_genre")
joblib.dump(le, "modelo_peru_genre/label_encoder.joblib")   # por compatibilidad extra
print("Modelo AST guardado en 'modelo_peru_genre/'")


In [ ]:
# Autenticarse en Hugging Face
from huggingface_hub import login

login()

## 9. Probar la inferencia descargando el modelo desde Hugging Face

In [ ]:
username = "pollitoconpapass"  # -> mi usuario de HF (cambialo x el tuyo ctm)
repo_name = "genre-classifier-peruvian-music"
repo_id = f"{username}/{repo_name}"

api = HfApi()

try:
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repositorio creado exitosamente en: https://huggingface.co/{repo_id}")

    print("Subiendo archivos al Hub...")
    api.upload_folder(
        folder_path="modelo_peru_genre",
        repo_id=repo_id,
        repo_type="model"
    )
    print("¡Modelo subido con éxito a Hugging Face Hub!")
except Exception as e:
    print("Ocurrió un error al subir el modelo:", e)
    print("Asegúrate de haber ingresado un token con permisos de ESCRITURA (write) y que tu nombre de usuario sea el correcto.")

In [ ]:
from huggingface_hub import hf_hub_download

print(f"Descargando componentes desde {repo_id}...")
le_path = hf_hub_download(repo_id=repo_id, filename="label_encoder.joblib")
le_loaded = joblib.load(le_path)

ast_loaded = AutoModelForAudioClassification.from_pretrained(repo_id).to(device)
fx_loaded = AutoFeatureExtractor.from_pretrained(repo_id)
print("Modelo AST descargado y cargado correctamente.")


In [ ]:
def predecir_con_modelo_hf(ruta_audio, max_seg=15):
    y, _ = librosa.load(ruta_audio, sr=SR, mono=True, duration=max_seg)
    if len(y) < CLIP_SAMPLES:
        y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
    clips = ventanas_inferencia(y)
    X_inf = np.stack([fx_loaded(c, sampling_rate=SR)['input_values'][0] for c in clips]).astype(np.float32)
    ast_loaded.eval()
    with torch.no_grad():
        logits = ast_loaded(input_values=torch.from_numpy(X_inf).to(device)).logits
    proba = torch.softmax(logits, dim=1).mean(0).cpu().numpy()
    orden = np.argsort(proba)[::-1]
    return str(le_loaded.classes_[orden[0]]), {str(le_loaded.classes_[i]): round(float(proba[i]), 3) for i in orden}


In [ ]:
# Main
archivo_audio = "clip_prueba.wav"   # <- pon aquí tu clip de 10–15 s
genero_predicho, probabilidades = predecir_con_modelo_hf(archivo_audio)
print("Género predicho:", genero_predicho)
print("Probabilidades:", probabilidades)
